In [ ]:
# ============================================================
# GLOBAL CRS / RHIZOMATIC GRAPH CONSTRUCTION
# Using local semantic filtering + global aggregation
# Final backbone: w >= 20
# ============================================================

import pandas as pd
import networkx as nx
import itertools
import ast
import os
import pyvis
from pyvis.network import Network

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ============================================================
# 1) LOAD DATA
# ============================================================

PATH_CSV = "YOURCSV"

df = pd.read_csv(PATH_CSV, low_memory=False)

def parse_keywords(x):
    if pd.isna(x) or x == "" or x == "[]":
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            val = ast.literal_eval(x)
            return val if isinstance(val, list) else []
        except Exception:
            return []
    return []

df["keywords_llm"] = df["keywords_llm"].apply(parse_keywords)
df = df[df["keywords_llm"].apply(len) > 0].reset_index(drop=True)

print(f"Articles with valid keywords: {len(df):,}")

# ============================================================
# 2) EMBEDDING MODEL AND PARAMETERS
# ============================================================

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
SIM_THRESHOLD = 0.4
W_BACKBONE = 20

model = SentenceTransformer(MODEL_NAME)

# ============================================================
# 3) GLOBAL RHIZOMATIC GRAPH CONSTRUCTION
#    From locally filtered semantic graphs
# ============================================================

G_rizoma = nx.Graph()

for idx, kws in enumerate(df["keywords_llm"].tolist(), start=1):

    # Basic normalization
    kws = [
        k.strip().lower()
        for k in kws
        if isinstance(k, str) and k.strip()
    ]

    # Remove duplicate keywords within the same document
    kws = sorted(set(kws))

    if len(kws) == 0:
        continue

    # Add nodes and update document frequency
    for k in kws:
        if G_rizoma.has_node(k):
            G_rizoma.nodes[k]["doc_freq"] += 1
        else:
            G_rizoma.add_node(k, doc_freq=1)

    if len(kws) < 2:
        continue

    # Compute document-level embeddings
    emb = model.encode(kws, normalize_embeddings=True)
    S = cosine_similarity(emb)

    # Local edges: co-occurrence + semantic similarity >= threshold
    for i, j in itertools.combinations(range(len(kws)), 2):
        sim = float(S[i, j])

        if sim >= SIM_THRESHOLD:
            a, b = kws[i], kws[j]

            if G_rizoma.has_edge(a, b):
                # Global weight: number of supporting documents
                G_rizoma[a][b]["weight"] += 1

                # Auxiliary similarity attributes
                G_rizoma[a][b]["sim_sum"] += sim
                G_rizoma[a][b]["sim_count"] += 1
            else:
                G_rizoma.add_edge(
                    a,
                    b,
                    weight=1,
                    sim_sum=sim,
                    sim_count=1
                )

    if idx % 5000 == 0:
        print(f"Processed {idx:,} articles...")

# Compute mean semantic similarity as an auxiliary attribute
for u, v, d in G_rizoma.edges(data=True):
    d["sim_mean"] = d["sim_sum"] / d["sim_count"]

print("\nGlobal rhizomatic graph constructed")
print(f"|V| = {G_rizoma.number_of_nodes():,}")
print(f"|E| = {G_rizoma.number_of_edges():,}")

# ============================================================
# 4) FINAL BACKBONE w >= 20
# ============================================================

H = nx.Graph()
H.add_nodes_from(G_rizoma.nodes(data=True))

H.add_edges_from([
    (u, v, d)
    for u, v, d in G_rizoma.edges(data=True)
    if float(d.get("weight", 1)) >= W_BACKBONE
])

# Remove isolated nodes after filtering
H.remove_nodes_from([n for n in list(H.nodes()) if H.degree(n) == 0])

print(f"\nFinal backbone w >= {W_BACKBONE}")
print(f"|V| = {H.number_of_nodes():,}")
print(f"|E| = {H.number_of_edges():,}")
print(f"Connected components = {nx.number_connected_components(H):,}")

if H.number_of_nodes() > 0:
    lcc_nodes = max(nx.connected_components(H), key=len)
    L = H.subgraph(lcc_nodes).copy()

    print(f"LCC |V| = {L.number_of_nodes():,}")
    print(f"LCC |E| = {L.number_of_edges():,}")
    print(f"LCC proportion = {L.number_of_nodes() / H.number_of_nodes():.4f}")

# ============================================================
# 5) LOUVAIN COMMUNITIES
# ============================================================

try:
    import community as community_louvain

    part = community_louvain.best_partition(
        H,
        weight="weight",
        random_state=42
    )

    nx.set_node_attributes(H, part, "community")
    print("\nLouvain communities detected")

except Exception as e:
    print("\nUnable to apply Louvain.")
    print(e)

    part = {n: 0 for n in H.nodes()}
    nx.set_node_attributes(H, part, "community")

# ============================================================
# 6) PYVIS VISUALIZATION OF THE w >= 20 BACKBONE
# ============================================================

OUT_HTML = "/Users/juanpablovargasherrera/Desktop/rizoma_final_w20_semantic_local.html"

palette = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
    "#393b79", "#637939", "#8c6d31", "#843c39", "#7b4173"
]

def color_for_comm(c):
    return palette[int(c) % len(palette)]

net = Network(
    height="900px",
    width="100%",
    bgcolor="white",
    font_color="black",
    notebook=False,
    cdn_resources="in_line"
)

net.force_atlas_2based(
    gravity=-50,
    central_gravity=0.01,
    spring_length=120,
    spring_strength=0.02,
    damping=0.4
)

# Nodes
for n, data in H.nodes(data=True):
    doc_freq = int(data.get("doc_freq", 1))
    comm = int(data.get("community", 0))
    degree = H.degree(n)

    size = 6 + 0.35 * (doc_freq ** 0.5)

    net.add_node(
        n,
        label=n,
        value=float(size),
        color=color_for_comm(comm),
        title=(
            f"{n}<br>"
            f"community: {comm}<br>"
            f"doc_freq: {doc_freq}<br>"
            f"degree: {degree}"
        )
    )

# Edges
for u, v, d in H.edges(data=True):
    w = float(d.get("weight", 1))
    sim_mean = float(d.get("sim_mean", 0))

    net.add_edge(
        u,
        v,
        value=w,
        title=(
            f"weight: {w}<br>"
            f"mean semantic similarity: {sim_mean:.3f}"
        )
    )

# Fix PyVis template
tpl = os.path.join(os.path.dirname(pyvis.__file__), "templates", "template.html")
net.set_template(tpl)

net.write_html(OUT_HTML, open_browser=False)

print(f"\nVisualization saved to:")
print(OUT_HTML)